# Market-1501 Interactive Guide
This notebook is the interactive version of `src/data/market1501_examples.py`.

It is designed for **EDA + understanding** before model training, with explanations before each runnable section.

## Project Context (PFE)
- Goal: build reliable person Re-ID foundations for your broader protest-leader detection system.
- This notebook helps validate data assumptions and loading logic before OSNet training.

## 1) Environment Setup
This notebook expects:
- Python 3.9+
- `Pillow`
- `numpy`
- Optional: `torch`, `torchvision` (for DataLoader and triplets)

If a package is missing, install from terminal:
- `pip install -r requirements.txt`
- or `pip install Pillow numpy torch torchvision`

This cell only checks availability and versions (safe to run multiple times).

In [1]:
import sys
import importlib

required = ["PIL", "numpy"]
optional = ["torch", "torchvision"]

print(f"Python: {sys.version.split()[0]}")

for pkg in required + optional:
    spec = importlib.util.find_spec(pkg)
    status = "OK" if spec else "MISSING"
    print(f"{pkg:12s}: {status}")

Python: 3.12.2
PIL         : OK
numpy       : OK
torch       : OK
torchvision : OK


## 2) Import Libraries
This section imports helper functions from `market1501_dataset.py` and defines project paths.

Why these imports matter:
- `parse_filename`: validate file naming conventions
- `load_dataset_split` + `get_dataset_statistics`: quick dataset EDA
- `group_by_identity`: identity distribution analysis
- `Market1501Dataset` / `TripletMarket1501`: practical PyTorch training input

In [2]:
from pathlib import Path
import sys

# Make src importable when running notebook from src/data
project_root = Path.cwd().parents[1] if (Path.cwd() / "market1501_examples.py").exists() else Path.cwd()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from data.market1501_dataset import (
    parse_filename,
    load_dataset_split,
    get_dataset_statistics,
    group_by_identity,
)

try:
    from data.market1501_dataset import Market1501Dataset, TripletMarket1501
    HAS_DATASETS = True
except Exception as exc:
    HAS_DATASETS = False
    print(f"PyTorch dataset classes unavailable: {exc}")

DATASET_ROOT = Path(r"C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15")
print(f"Project root: {project_root}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Exists: {DATASET_ROOT.exists()}")

Project root: c:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour
Dataset root: C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15
Exists: True


## 3) Load or Create Data
We load Market-1501 from your local path and inspect the canonical splits:
- `bounding_box_train`
- `query`
- `bounding_box_test`

This confirms dataset integrity before any training work.

In [3]:
train_dir = DATASET_ROOT / "bounding_box_train"
query_dir = DATASET_ROOT / "query"
gallery_dir = DATASET_ROOT / "bounding_box_test"

for name, folder in [("train", train_dir), ("query", query_dir), ("gallery", gallery_dir)]:
    print(f"{name:8s}: {folder} -> {'OK' if folder.exists() else 'MISSING'}")

splits = {}
if train_dir.exists():
    splits["train"] = load_dataset_split(str(train_dir))
if query_dir.exists():
    splits["query"] = load_dataset_split(str(query_dir))
if gallery_dir.exists():
    splits["gallery"] = load_dataset_split(str(gallery_dir))

print("\nLoaded splits:")
for k, v in splits.items():
    print(f"- {k}: {len(v):,} images")

train   : C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15\bounding_box_train -> OK
query   : C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15\query -> OK
gallery : C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15\bounding_box_test -> OK

Loaded splits:
- train: 12,936 images
- query: 3,368 images
- gallery: 13,115 images


## 4) Prepare Inputs
Before training, we verify metadata parsing and basic preprocessing assumptions:
- filename parsing returns valid fields
- split statistics are coherent
- identities can be grouped for downstream sampling

PFE mapping: this section supports the Dataset/Preprocessing chapter.

In [4]:
# Parse one canonical filename
sample_name = "0001_c1s1_001051_00.jpg"
info = parse_filename(sample_name)

print("Parsed sample filename:")
if info is None:
    print("- Could not parse sample name")
else:
    print(f"- person_id    : {info.person_id}")
    print(f"- camera_id    : {info.camera_id}")
    print(f"- sequence_id  : {info.sequence_id}")
    print(f"- frame_id     : {info.frame_id}")
    print(f"- detection_idx: {info.detection_idx}")

# Show train statistics if available
if "train" in splits:
    stats = get_dataset_statistics(splits["train"])
    print("\nTrain split statistics:")
    for k, v in stats.items():
        print(f"- {k}: {v}")
else:
    print("\nTrain split not loaded; check DATASET_ROOT.")

Parsed sample filename:
- person_id    : 1
- camera_id    : 1
- sequence_id  : 1
- frame_id     : 1051
- detection_idx: 0

Train split statistics:
- total_images: 12936
- num_identities: 751
- num_cameras: 6
- camera_ids: [1, 2, 3, 4, 5, 6]
- avg_images_per_person: 17.22503328894807
- min_images_per_person: 2
- max_images_per_person: 72
- avg_cameras_per_person: 4.343541944074567


## 5) Implement Core Logic
This section executes the main reusable workflows from your examples file:
1. Identity analysis (top/bottom identities by number of images)
2. PyTorch DataLoader creation (standard train batches)
3. Triplet loader creation (anchor/positive/negative)

PFE mapping: this is the practical bridge from EDA to model training.

In [5]:
# 5.1 Identity analysis
if "train" in splits and len(splits["train"]) > 0:
    by_identity = group_by_identity(splits["train"])
    id_counts = sorted(((pid, len(imgs)) for pid, imgs in by_identity.items()), key=lambda x: x[1], reverse=True)

    print("Top 5 identities (most images):")
    for pid, count in id_counts[:5]:
        cams = sorted({img.camera_id for img in by_identity[pid]})
        print(f"- ID {pid:04d}: {count} images, cameras={cams}")

    print("\nBottom 5 identities (least images):")
    for pid, count in id_counts[-5:]:
        cams = sorted({img.camera_id for img in by_identity[pid]})
        print(f"- ID {pid:04d}: {count} images, cameras={cams}")
else:
    print("Train split unavailable. Cannot run identity analysis.")

Top 5 identities (most images):
- ID 0139: 72 images, cameras=[1, 3, 4, 5, 6]
- ID 0105: 70 images, cameras=[1, 2, 3, 5, 6]
- ID 0272: 67 images, cameras=[1, 2, 3, 5, 6]
- ID 0208: 63 images, cameras=[1, 2, 3, 4, 5, 6]
- ID 0022: 59 images, cameras=[1, 2, 3, 4, 5, 6]

Bottom 5 identities (least images):
- ID 0864: 3 images, cameras=[2, 3]
- ID 1237: 3 images, cameras=[1, 2, 3]
- ID 0084: 2 images, cameras=[1, 5]
- ID 0129: 2 images, cameras=[2, 4]
- ID 1451: 2 images, cameras=[2, 3]


In [6]:
# 5.2 PyTorch DataLoader example
try:
    import torch
    from torch.utils.data import DataLoader
    from torchvision import transforms
    torch_ok = True
except Exception as exc:
    torch_ok = False
    print(f"PyTorch/torchvision unavailable: {exc}")

if torch_ok and HAS_DATASETS and DATASET_ROOT.exists():
    train_tfms = transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    ds = Market1501Dataset(root=str(DATASET_ROOT), split="train", transform=train_tfms)
    dl = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0)
    batch = next(iter(dl))

    print("DataLoader batch preview:")
    print(f"- image shape: {tuple(batch['image'].shape)}")
    print(f"- labels     : {batch['label'][:5]}")
    print(f"- person_ids : {batch['person_id'][:5]}")
else:
    print("Skipped DataLoader demo (missing packages/classes/dataset).")

Loaded train split: 12936 images, 751 identities
DataLoader batch preview:
- image shape: (8, 3, 256, 128)
- labels     : tensor([352, 613, 505,  78, 515])
- person_ids : tensor([ 665, 1193,  963,  160,  982])


In [7]:
# 5.3 Triplet loader example
if torch_ok and HAS_DATASETS and DATASET_ROOT.exists():
    triplet_ds = TripletMarket1501(root=str(DATASET_ROOT))
    triplet_dl = DataLoader(triplet_ds, batch_size=4, shuffle=True, num_workers=0)
    t_batch = next(iter(triplet_dl))

    print("Triplet batch preview:")
    print(f"- anchor shape  : {tuple(t_batch['anchor'].shape)}")
    print(f"- positive shape: {tuple(t_batch['positive'].shape)}")
    print(f"- negative shape: {tuple(t_batch['negative'].shape)}")
    print(f"- anchor_ids    : {t_batch['anchor_id'][:4]}")
    print(f"- negative_ids  : {t_batch['negative_id'][:4]}")
else:
    print("Skipped triplet demo (missing packages/classes/dataset).")

Triplet Dataset: 12936 images, 751 identities
Triplet batch preview:
- anchor shape  : (4, 3, 256, 128)
- positive shape: (4, 3, 256, 128)
- negative shape: (4, 3, 256, 128)
- anchor_ids    : tensor([  22, 1313,  370,  741])
- negative_ids  : tensor([584, 606, 385, 830])


## 6) Inspect and Visualize Results
Review these outputs:
- loaded image counts per split
- train statistics (`num_identities`, `num_cameras`, etc.)
- top/bottom identities by sample count
- tensor shapes from DataLoader and triplet loader

Interpretation:
- if counts/shapes look correct, your data pipeline is ready for OSNet training.

In [8]:
# Optional quick visualization table (text only, no extra deps)
summary_rows = []
for split_name in ["train", "query", "gallery"]:
    if split_name in splits:
        summary_rows.append((split_name, len(splits[split_name])))

print("Split summary:")
for name, count in summary_rows:
    print(f"- {name:8s}: {count:,}")

Split summary:
- train   : 12,936
- query   : 3,368
- gallery : 13,115


## 7) Validate Output
This cell performs lightweight sanity checks to ensure the notebook output is consistent with expected Market-1501 usage:
- dataset root exists
- at least one split loaded
- parsed filename was valid

In [9]:
errors = []
if not DATASET_ROOT.exists():
    errors.append("DATASET_ROOT does not exist")
if len(splits) == 0:
    errors.append("No splits loaded")
if info is None:
    errors.append("Filename parser returned None for canonical sample")

if errors:
    print("Validation issues:")
    for e in errors:
        print(f"- {e}")
else:
    print("Validation passed: basic data pipeline checks are OK.")

Validation passed: basic data pipeline checks are OK.


## 8) Save Results
This section saves lightweight EDA artifacts for your report:
- split counts
- a subset of train statistics

You can later reference these files in your PFE chapter on dataset preparation.

In [10]:
import json

outputs_dir = project_root / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

split_counts = {k: len(v) for k, v in splits.items()}
with open(outputs_dir / "market1501_split_counts.json", "w", encoding="utf-8") as f:
    json.dump(split_counts, f, indent=2)

train_stats = get_dataset_statistics(splits["train"]) if "train" in splits else {}
with open(outputs_dir / "market1501_train_stats.json", "w", encoding="utf-8") as f:
    json.dump(train_stats, f, indent=2)

print(f"Saved: {outputs_dir / 'market1501_split_counts.json'}")
print(f"Saved: {outputs_dir / 'market1501_train_stats.json'}")

Saved: c:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour\outputs\market1501_split_counts.json
Saved: c:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour\outputs\market1501_train_stats.json


## Next Step
This notebook is for **EDA + understanding + validation** of Market-1501 loading and preprocessing.

The actual model training should be implemented in a separate script (e.g., `src/training/train_reid.py`) that imports from `src/data/market1501_dataset.py`.

Recommended continuation:
1. Build OSNet training loop
2. Track CMC/mAP on query vs gallery
3. Save checkpoints and training curves